In [13]:
nums = [1, 2, 3, 4, 5, 6]
for i in range(1, len(nums)):
    nums[i] += nums[i-1]
queries = [(1, 3), (0, 5), (2, 4)]
nums = [0] + nums
for x, y in queries: 
    print(nums[y+1] - nums[x])

9
21
12


In [23]:
nums = [1, 1, 1]
k = 2
import collections

prefix = 0 
count = 0 
# sum --k how many times ?
seen = collections.defaultdict(int)
seen[0] = 1
for num in nums: 
    prefix += num 
    count += seen[prefix-k]
    seen[prefix] +=1
count 



2

In [35]:
nums = [1, 2, 3, 4]
prefix_product = [1]*len(nums)
suffix_product = [1]*len(nums)
for i in range(1, len(nums)):
    prefix_product[i] = nums[i-1] * prefix_product[i-1]

for i in range(len(nums)-2, -1, -1):
    suffix_product[i] = nums[i+1] * suffix_product[i+1]

res = []
for  i in range(len(nums)):
    res.append(prefix_product[i]* suffix_product[i])
res

[24, 12, 8, 6]

In [41]:
nums = [4, 5, 0, -2, -3, 1]
k = 5
seen = collections.defaultdict(int)
prefix_sum = 0 
count = 0 
seen[0] = 1
for x in nums: 
    prefix_sum += x
    prefix_sum = prefix_sum%k 
    
    count += seen[prefix_sum]
    seen[prefix_sum] +=1
count 

7

In [94]:
nums = [1, 7, 3, 6, 5, 6]
# pivot = left sum = right sum 
left_sum = 0 
total_sum  = sum(nums)
for i in range(len(nums)):
    right_sum = total_sum - left_sum - nums[i]
    if left_sum == right_sum : 
        print("pivot ", i )
        break 
    left_sum += nums[i]



pivot  3


In [104]:
matrix = [
    [3, 0, 1, 4, 2],
    [5, 6, 3, 2, 1],
    [1, 2, 0, 1, 5],
    [4, 1, 0, 1, 7],
    [1, 0, 3, 0, 5]
]
prefix = [[0]* (len(matrix[0])+1) for _ in range(len(matrix)+1)] 
for i in range(1 , len(matrix)+1):
    for j in range( 1 , len(matrix[0])+1):
        prefix[i][j] = prefix[i-1][j] + prefix[i][j-1] + matrix[i-1][j-1] - prefix[i-1][j-1]
prefix

[[0, 0, 0, 0, 0, 0],
 [0, 3, 3, 4, 8, 10],
 [0, 8, 14, 18, 24, 27],
 [0, 9, 17, 21, 28, 36],
 [0, 13, 22, 26, 34, 49],
 [0, 14, 23, 30, 38, 58]]

In [110]:
r1, c1 = 1, 1 
r2 , c2 = 3, 3, 
prefix[r2+1][c2+1] - prefix[r1][c2+1] - prefix[r2+1][c1] + prefix[r1][c1]


16

# Prefix Sum — All Tricks & Patterns

## Pattern 1: Range Sum Query
- Build prefix with leading zero: `prefix = [0] + cumulative_sums`
- Query: `sum(i, j) = prefix[j+1] - prefix[i]`
- O(n) build, O(1) per query

## Pattern 2: Subarray Sum Equals K (prefix + hashmap)
- Track running prefix sum
- At each step, look up `prefix - k` in hashmap → count matches
- `seen[0] = 1` to account for subarrays starting at index 0
- O(n) time, O(n) space
- **Why not sliding window?** Negative numbers break monotonicity — expanding can decrease sum, shrinking can increase it. No way to know which direction to go.

## Pattern 3: Subarray Sum Divisible by K
- Same as Pattern 2, but store `prefix % k`
- If two prefix sums have the same remainder, their difference is divisible by k
- Look up `seen[prefix % k]` — any match means divisible

## Pattern 4: Transform the Problem
- Equal 0s and 1s → replace 0 with -1 → find longest subarray with sum = 0
- Equal count of two characters → assign +1/-1 → prefix sum
- The clue: "equal number of X and Y in subarray" → transform to sum = 0

## Pattern 5: Product Except Self
- Prefix product from left: `prefix_product[i]` = product of everything before i
- Suffix product from right: `suffix_product[i]` = product of everything after i
- Answer: `prefix_product[i] * suffix_product[i]`
- No division needed

## Pattern 6: 2D Prefix Sum
- **Build** (with zero padding):
```python
prefix[i][j] = prefix[i-1][j] + prefix[i][j-1] - prefix[i-1][j-1] + matrix[i-1][j-1]
```
- **Query** rectangle (r1,c1) to (r2,c2):
```python
prefix[r2+1][c2+1] - prefix[r1][c2+1] - prefix[r2+1][c1] + prefix[r1][c1]
```
- Inclusion-exclusion: total - top - left + top_left (double-subtracted corner)

## Pattern 7: Difference Array (inverse of prefix sum)
- **Problem:** Multiple range updates `[x, y] += val`, then read final values
- **Trick:** Store only the "start" and "end" of each update:
```python
diff[x] += val       # start adding from x
diff[y+1] -= val     # stop adding after y
```
- **Reconstruct** actual values by taking prefix sum of diff array
- O(1) per update, O(n) to reconstruct
- **Why it works:** +val at x starts the effect, -val at y+1 cancels it. Prefix sum carries the effect through all indices in between.

```python
# Example: add 5 to indices 2..4 in array of length 7
diff = [0, 0, +5, 0, 0, -5, 0]
prefix_sum(diff) → [0, 0, 5, 5, 5, 0, 0]  # actual values
```

---

## When to recognize prefix sum problems
- "Subarray sum" → prefix + hashmap
- "Range query" → prefix array
- "Range update" → difference array
- "Product except self" → prefix/suffix products
- "Equal count of X and Y" → transform to +1/-1, prefix sum = 0
- "2D rectangle sum" → 2D prefix with inclusion-exclusion